# RB Pipeline v4 Launcher v0.5

Use this notebook to run `detect -> silhouette -> pack_dual_stream` with edge-based ROI detection (v1-style) as the default detector backend. The notebook remains a control pane; reusable preprocessing behavior lives in `rb_pipeline_v4`.

Brightness normalization controls are surfaced in their own generated launcher pane. The implementation lives in `rb_pipeline_v4.brightness_normalization` and is wired through `PackDualStreamStageConfigV4`.


In [ ]:
from pathlib import Path

import importlib
import sys
from IPython.display import display


def find_pipeline_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    if candidate.is_file():
        candidate = candidate.parent

    for current in (candidate, *candidate.parents):
        if (current / 'rb_pipeline_v4').exists() and (current / 'input-images').exists():
            return current

        nested = current / '02_synthetic-data-processing-v4.0'
        if (nested / 'rb_pipeline_v4').exists() and (nested / 'input-images').exists():
            return nested

    raise RuntimeError('Could not locate 02_synthetic-data-processing-v4.0 root.')


PIPELINE_ROOT = find_pipeline_root()
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

import rb_pipeline_v4.brightness_normalization as brightness_mod
import rb_pipeline_v4.config as config_mod
import rb_pipeline_v4.widgets as widgets_base_mod
import rb_pipeline_v4.widgets_v05 as widgets_mod

# Keep this launcher honest during iterative UI work in a live kernel.
# Plain imports can otherwise keep displaying an old PipelineLauncherV4 class.
for module in (config_mod, brightness_mod, widgets_base_mod, widgets_mod):
    importlib.reload(module)

actual_build = getattr(widgets_mod, 'WIDGETS_UI_BUILD_V05', 'unknown')
print('Loaded widgets module:', widgets_mod.__file__)
print('UI build:', actual_build)

_previous_widget = globals().get('_pipeline_launcher_v05_widget')
if _previous_widget is not None:
    try:
        _previous_widget.close()
    except Exception:
        pass

launcher = widgets_mod.PipelineLauncherV5(PIPELINE_ROOT)
launcher_widget = launcher.widget
display(launcher_widget)
globals()['_pipeline_launcher_v05_instance'] = launcher
globals()['_pipeline_launcher_v05_widget'] = launcher_widget

print('Preview controls:', type(launcher.preview_sample_offset_input).__name__, '+', type(launcher.preview_refresh_button).__name__)
print('Brightness controls:', type(launcher.brightness_norm_enabled_checkbox).__name__, '+', type(launcher.brightness_norm_method_dropdown).__name__)
